In [ ]:
from typing import Any, cast
from letta_client import Letta
from prompts.persona import PERSONAS, HUMAN_TEMPLATE
from prompts.system_prompts import CUSTOM_V1_PROMPT
from utils.message_parser import chat_and_print

client = Letta(base_url="http://localhost:8283")

In [ ]:
memory_blocks = cast(Any, [
    {
        "label": "persona",
        "value": PERSONAS["linxiaotang"],
    },
    {
        "label": "human",
        "value": HUMAN_TEMPLATE,
    },
])

agent = client.agents.create(
    name="qwen-agent20260327",
    system=CUSTOM_V1_PROMPT,
    model="lmstudio_openai/qwen3.5-27b",
    # model="lmstudio_openai/qwen/qwen3.5-35b-a3b",
    timezone="Asia/Shanghai",
    context_window_limit=16384,
    memory_blocks=memory_blocks,
)

In [ ]:
response = chat_and_print(
    client=client,
    agent_id=agent.id,
    input="你好，我是张伟"
)

In [ ]:
response = chat_and_print(
    client=client,
    agent_id=agent.id,
    input="你好，我喜欢狗狗，你呢？"
)

In [ ]:
# Inspect what this running Letta server can actually use right now.
all_models = list(client.models.list())
all_embeddings = list(client.models.embeddings.list())

embedding_handles_set = {e.handle for e in all_embeddings if getattr(e, "handle", None)}

# models.list() may include embedding-like handles too (e.g. doubao-embedding-*).
# We classify those by name and move them to embedding handles.
available_llm_handles = []
for model in all_models:
    handle = getattr(model, "handle", "")
    model_type = getattr(model, "api_model_type", None) or getattr(model, "model_type", None)
    if not handle:
        continue
    lowered = handle.lower()
    if "embedding" in lowered or "embed" in lowered:
        embedding_handles_set.add(handle)
        continue
    if handle in embedding_handles_set:
        continue
    if model_type and model_type != "llm":
        continue
    available_llm_handles.append(handle)

available_llm_handles = sorted(set(available_llm_handles))
available_embedding_handles = sorted(embedding_handles_set)

print("LLM handles (filtered):")
for h in available_llm_handles:
    print(f"- {h}")

print("\nEmbedding handles:")
for h in available_embedding_handles:
    print(f"- {h}")

model_type_by_handle = {
    m.handle: (getattr(m, "api_model_type", None) or getattr(m, "model_type", None))
    for m in all_models
    if getattr(m, "handle", None)
}
embedding_type_by_handle = {
    e.handle: (getattr(e, "api_model_type", None) or getattr(e, "model_type", None))
    for e in all_embeddings
    if getattr(e, "handle", None)
}

overlap = sorted(set(model_type_by_handle) & set(embedding_type_by_handle))
if overlap:
    print("\nHandles present in BOTH models.list() and embeddings.list():")
    for h in overlap:
        print(f"- {h}: models.list() type={model_type_by_handle[h]!r}, embeddings.list() type={embedding_type_by_handle[h]!r}")

In [ ]:
def create_agent_with_handles(
    model_handle: str,
    embedding_handle: str | None = None,
    name: str = "dev-agent",
    system_prompt: str = CUSTOM_V1_PROMPT,
):
    memory_blocks = cast(Any, [
        {
            "label": "persona",
            "value": PERSONAS["linxiaotang"],
        },
        {
            "label": "human",
            "value": HUMAN_TEMPLATE,
        },
    ])

    if embedding_handle:
        return client.agents.create(
            name=name,
            system=system_prompt,
            model=model_handle,
            embedding=embedding_handle,
            timezone="Asia/Shanghai",
            context_window_limit=16384,
            memory_blocks=memory_blocks,
        )

    return client.agents.create(
        name=name,
        system=system_prompt,
        model=model_handle,
        timezone="Asia/Shanghai",
        context_window_limit=16384,
        memory_blocks=memory_blocks,
    )

In [ ]:
# Example A: local Qwen + local embedding (works in current setup)
agent_local = create_agent_with_handles(
    model_handle="lmstudio_openai/qwen3.5-27b",
    embedding_handle="lmstudio_openai/text-embedding-qwen3-embedding-0.6b",
    name="dev-local-qwen",
)
print("Created local agent:", agent_local.id)

# Example B: Doubao model handle (only works if the handle is registered in Letta)
# This requires Doubao/ARK provider config to be available on the Letta server.
try:
    agent_doubao = create_agent_with_handles(
        model_handle="openai-proxy/doubao-seed-1-8-251228",
        embedding_handle="lmstudio_openai/text-embedding-qwen3-embedding-0.6b",
        name="dev-doubao",
    )
    print("Created doubao agent:", agent_doubao.id)
except Exception as exc:
    print("Doubao handle not available on current Letta server:")
    print(exc)

In [ ]:
# Recommended approach: do NOT pass llm_config when creating agents.
# Use registered model/embedding handles so Letta resolves provider settings internally.
from pprint import pprint

def create_agent_with_registered_handles(name: str, model_handle: str, embedding_handle: str | None = None):
    memory_blocks = cast(Any, [
        {"label": "persona", "value": PERSONAS["linxiaotang"]},
        {"label": "human", "value": HUMAN_TEMPLATE},
    ])

    if embedding_handle:
        return client.agents.create(
            name=name,
            system=CUSTOM_V1_PROMPT,
            model=model_handle,
            embedding=embedding_handle,
            timezone="Asia/Shanghai",
            context_window_limit=16384,
            memory_blocks=memory_blocks,
        )

    return client.agents.create(
        name=name,
        system=CUSTOM_V1_PROMPT,
        model=model_handle,
        timezone="Asia/Shanghai",
        context_window_limit=16384,
        memory_blocks=memory_blocks,
    )

# Doubao by model handle (works when the handle is registered on the Letta server)
try:
    agent_doubao = create_agent_with_registered_handles(
        name="dev-doubao-handle",
        model_handle="openai-proxy/doubao-seed-1-8-251228",
        embedding_handle="lmstudio_openai/text-embedding-qwen3-embedding-0.6b",
    )
    pprint({"id": agent_doubao.id, "model": agent_doubao.model})
except Exception as exc:
    print("Doubao handle creation failed (usually handle not registered or credentials invalid):")
    print(exc)